# Atomic-GDN2: exact arithmetic confirmation on TPU

**Akseleu Omirbay.** Новая среда **Colab TPU v5e-1 → Run all**. Данные и скрипт встроены; токены/Drive не нужны.

## Что подтверждает этот запуск
Ранее сохранённые входы и cotangent реальной полной LM заново обрабатываются на TPU. Все **27 перекрёстных сумм** OLD/JAX_REF/NEW × три сида сравниваются с независимым Fraction-эталоном. Дополнительно: девять точных проверок перестановки слагаемых, девять device-разложений разности, контрольные случаи с cancellation, subnormals и произведениями, переполняющими FP32, и isolated TPU convolution backward для девяти захватов. Проверяется ранее рассчитанная доля **98.2542%**.

### Как считаем точно на TPU
FP32 входы разбираются на знак, целую мантиссу и двоичную степень. Произведения считаются через байты uint32, затем суммируются в знаковых int32 bins с основанием 256. Для N=16384 консервативная граница модуля суммы одного bin: `N*27*255 = 112803840 < 2^31`. Для проверяемых разностей/сумм разностей граница также безопасна. Используются 80 bins со стартовой степенью -320. Декодирование в Python Fraction служит только чтением точного device-результата. **CPU fallback не разрешён**: ноутбук завершает проверку ошибкой, если backend не TPU. Полный рабочий код сохраняется в ZIP.

### Объём вывода
Это **новые вычисления на TPU над зафиксированными захватами**. Полная LM и Pallas backward повторно здесь не запускаются; их данные происходят из приложенного проверенного архива. Isolated convolution replay имеет другой граф XLA, поэтому его FP32 результат может отличаться от исходной полной модели. Точная сумма сохранённых факторов не равна точному неокруглённому градиенту всей LM. Успешный результат подтверждает эти расчёты, а не универсальную корректность ядра для всех входов.

На выходе: исходный архив захватов, новые int32 bins, таблицы, JSON, полные логи, версии пакетов и SHA256. При любой ошибке сохраняется статус failed и доступные данные. Пришлите итоговый ZIP.


In [ ]:
import sys,subprocess,pathlib,json,hashlib,zipfile,base64,datetime,os
RUN_ID=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
WORK=pathlib.Path('/content')/('atomic-exact-tpu-'+RUN_ID);WORK.mkdir();OUT=WORK/'results';OUT.mkdir()
subprocess.check_call([sys.executable,'-m','pip','install','-q','virtualenv'])
subprocess.check_call([sys.executable,'-m','virtualenv',str(WORK/'venv')])
PY=str(WORK/'venv/bin/python')
PACKAGES=['jax[tpu]==0.11.2','jaxlib==0.11.2','libtpu==0.0.48.1','numpy==2.1.3','scipy==1.16.3','ml_dtypes==0.6.0','opt_einsum==3.4.0']
subprocess.check_call([PY,'-m','pip','install','-q',*PACKAGES])
(OUT/'pip-freeze.txt').write_text(subprocess.check_output([PY,'-m','pip','freeze'],text=True))


In [ ]:
CAPTURE_B64='...'
SOURCE='"""Device-executed exact binary32 dot audit of archived real-LM captures."""\nimport argparse,csv,hashlib,io,json,math,os,platform,sys,traceback,zipfile\nfrom fractions import Fraction\nfrom decimal import Decimal,localcontext\nfrom pathlib import Path\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\njax.config.update(\'jax_enable_x64\',False)\njax.config.update(\'jax_default_matmul_precision\',\'highest\')\nMIN_EXP=-320\nN_BINS=80\n\n@jax.jit\ndef exact_device_bins(x,d):\n """No floating multiplication: base-256 limbs, signed int32 scatter accumulation."""\n xb=jax.lax.bitcast_convert_type(x.reshape(-1),jnp.uint32)\n db=jax.lax.bitcast_convert_type(d.reshape(-1),jnp.uint32)\n ex=((xb>>23)&255).astype(jnp.int32);ed=((db>>23)&255).astype(jnp.int32)\n mx=(xb&0x7fffff)|jnp.where(ex==0,jnp.uint32(0),jnp.uint32(1<<23))\n md=(db&0x7fffff)|jnp.where(ed==0,jnp.uint32(0),jnp.uint32(1<<23))\n ex=jnp.where(ex==0,-149,ex-150);ed=jnp.where(ed==0,-149,ed-150)\n sign=1-2*((xb^db)>>31).astype(jnp.int32)\n result=jnp.zeros(N_BINS,jnp.int32)\n for i in range(3):\n  for j in range(3):\n   a=(mx>>(8*i))&255;b=(md>>(8*j))&255\n   shift=ex+ed-MIN_EXP+8*(i+j);index=shift//8\n   # Each uint32 product is at most 255^2; shifting by <=7 stays below 2^23.\n   p=(a*b)<<(shift%8).astype(jnp.uint32)\n   for k in range(3):\n    byte=((p>>(8*k))&255).astype(jnp.int32)*sign\n    result=result.at[index+k].add(byte)\n return result\n\ndef decode_device_bins(bins):\n integer=sum(int(v)<<(8*i) for i,v in enumerate(np.asarray(bins)))\n return Fraction(integer,1<<(-MIN_EXP))\n\ndef fraction_oracle(x,d):\n return sum((Fraction.from_float(float(a))*Fraction.from_float(float(b)) for a,b in zip(x.ravel(),d.ravel())),Fraction())\n\ndef decimal(f):\n with localcontext() as ctx:\n  ctx.prec=100;return str(Decimal(f.numerator)/Decimal(f.denominator))\n\ndef check_device(x,d,allow_cpu=False):\n assert x.shape==d.shape and x.dtype==d.dtype==np.float32\n assert np.isfinite(x).all() and np.isfinite(d).all()\n # Conservative worst case counts all 27 byte contributions in one bin.\n assert x.size*27*255<2**31,\'int32 overflow bound exceeded\'\n xx=jax.device_put(x);dd=jax.device_put(d)\n for original,device in ((x,xx),(d,dd)):\n  bits=jax.lax.bitcast_convert_type(device,jnp.uint32)\n  assert np.array_equal(np.asarray(bits),original.view(np.uint32)),\'Input bits changed on device transfer\'\n bins=exact_device_bins(xx,dd);bins.block_until_ready()\n if not allow_cpu:assert all(device.platform==\'tpu\' for device in bins.devices())\n exact=decode_device_bins(bins);ref=fraction_oracle(x,d)\n assert exact==ref,\'Exact device sum differs from independent Fraction oracle\'\n return bins,exact\n\n@jax.jit\ndef local_conv_gradient(u,d):\n """Isolated selected-channel convolution, not a fresh full-LM backward."""\n def loss(w):\n  out=jax.lax.conv_general_dilated(u[...,None],w[:,None,None],(1,),[(3,0)],\n   dimension_numbers=(\'NHC\',\'HIO\',\'NHC\'),precision=jax.lax.Precision.HIGHEST)\n  return jnp.sum(out[...,0]*d)\n return jax.grad(loss)(jnp.zeros(4,jnp.float32))\n\ndef dump(out,report):\n p=out/\'confirmation.json\';tmp=p.with_suffix(\'.tmp\');tmp.write_text(json.dumps(report,indent=2,allow_nan=False));tmp.replace(p)\n\ndef run(a,r):\n backend=jax.default_backend()\n if not a.allow_cpu:assert backend==\'tpu\',f\'TPU required, got {backend}\'\n r.update(backend=backend,devices=[str(d) for d in jax.devices()],jax=jax.__version__,numpy=np.__version__,\n  python=sys.version,platform=platform.platform(),exact_algorithm=\'IEEE binary32 bit decomposition; uint32 byte products; signed int32 base-256 scatter bins; host decoding only\',\n  safety={\'minimum_exponent\':MIN_EXP,\'bins\':N_BINS,\'max_terms\':16384,\'conservative_int32_bin_bound\':16384*27*255,\'limit_exclusive\':2**31},\n  scope=\'Fresh device arithmetic and isolated convolution replay on frozen full-LM captures. No new full-LM forward/backward or speed benchmark.\')\n dump(a.out,r)\n tiny=np.nextafter(np.float32(0),np.float32(1));big=np.finfo(np.float32).max\n controls=[(\'zero\',np.zeros(16,np.float32),np.ones(16,np.float32)),\n  (\'cancellation_to_one\',np.array([2**25,1,-2**25],np.float32),np.ones(3,np.float32)),\n  (\'subnormal_product\',np.array([tiny],np.float32),np.array([tiny],np.float32)),\n  (\'overflowing_float32_products_cancel\',np.array([big,big,1],np.float32),np.array([big,-big,1],np.float32))]\n rng=np.random.default_rng(20261017)\n for i in range(4):\n  bits=rng.integers(0,2**32,size=(2,256),dtype=np.uint32)\n  bits=np.where(((bits>>23)&255)==255,bits^np.uint32(1<<23),bits).astype(np.uint32)\n  controls.append((f\'finite_binary32_bits_{i}\',bits[0].view(np.float32),bits[1].view(np.float32)))\n for label,x,d in controls:\n  _,v=check_device(x,d,a.allow_cpu);r[\'controls\'].append({\'case\':label,\'exact_equal\':True,\'exact_value\':decimal(v)})\n  print(\'EXACT_CONTROL_PASS\',label,flush=True);dump(a.out,r)\n expected_hash=\'dd6854993c17d8f8a7e3fffdd3804d4c1481636c84768bb62ec2641090b615b6\'\n assert hashlib.sha256(a.capture.read_bytes()).hexdigest()==expected_hash\n data={};rows={}\n with zipfile.ZipFile(a.capture) as z:\n  manifest=json.loads(z.read(\'manifest.json\'))[\'files\']\n  assert all(hashlib.sha256(z.read(n)).hexdigest()==h for n,h in manifest.items())\n  for path in (\'OLD\',\'JAX_REF\',\'NEW\'):\n   record=json.loads(z.read(f\'results/{path}-probe.json\'));assert record[\'status\']==\'completed\'\n   for row in record[\'seeds\']:\n    key=path,row[\'seed\'];rows[key]=row\n    blob=z.read(\'results/\'+row[\'arrays\']);assert hashlib.sha256(blob).hexdigest()==row[\'arrays_sha256\']\n    with np.load(io.BytesIO(blob),allow_pickle=False) as n:data[key]={k:n[k].copy() for k in n.files}\n r[\'capture_archive_sha256\']=expected_hash\n saved_bins={};exact={};csvrows=[]\n for seed in (0,1,2):\n  for x_path in (\'OLD\',\'JAX_REF\',\'NEW\'):\n   for d_path in (\'OLD\',\'JAX_REF\',\'NEW\'):\n    x=data[x_path,seed][\'aligned_input\'];d=data[d_path,seed][\'cotangent\']\n    bins,value=check_device(x,d,a.allow_cpu);key=seed,x_path,d_path;saved_bins[key]=bins;exact[key]=value\n    row={\'seed\':seed,\'input_path\':x_path,\'cotangent_path\':d_path,\'exact_equal\':True,\'exact_contraction\':decimal(value)}\n    r[\'cross_contractions\'].append(row);csvrows.append(row)\n    if x_path==d_path:\n     permutation=rng.permutation(x.size)\n     shuffled,other=check_device(x.ravel()[permutation],d.ravel()[permutation],a.allow_cpu)\n     assert np.array_equal(np.asarray(shuffled),np.asarray(bins)) and other==value\n     u=data[x_path,seed][\'conv_input\'];got=local_conv_gradient(jax.device_put(u),jax.device_put(d));got.block_until_ready()\n     assert np.isfinite(np.asarray(got)).all()\n     r[\'local_float_replays\'].append({\'seed\':seed,\'path\':x_path,\'archived_full_LM_gradient\':rows[x_path,seed][\'raw_coordinate_gradient\'],\n      \'isolated_TPU_conv_gradient\':float(got[2]),\'exact_frozen_gradient\':decimal(value),\n      \'isolated_gradient_absolute_error\':float(abs(Fraction.from_float(float(got[2]))-value)),\n      \'order_permutation_exact_equal\':True,\'note\':\'Different compiled graph from full LM; floating results need not be bitwise identical.\'})\n    print(\'CAPTURE_EXACT_PASS\',key,flush=True);dump(a.out,r)\n  for aa,bb in [(\'OLD\',\'NEW\'),(\'OLD\',\'JAX_REF\'),(\'JAX_REF\',\'NEW\')]:\n   A=saved_bins[seed,aa,aa];BA=saved_bins[seed,bb,aa];B=saved_bins[seed,bb,bb]\n   # Bin subtractions and equality are executed on the selected device; bound stays <2^31.\n   dx=jax.jit(lambda a,b:a-b)(A,BA);dd=jax.jit(lambda a,b:a-b)(BA,B)\n   total=jax.jit(lambda a,b:a-b)(A,B)\n   assert bool(jax.jit(lambda a,b,c:jnp.all(a+b==c))(dx,dd,total))\n   assert decode_device_bins(total)==exact[seed,aa,aa]-exact[seed,bb,bb]\n   r[\'decomposition\'].append({\'seed\':seed,\'path_a\':aa,\'path_b\':bb,\'exact_device_identity\':True,\n    \'input_term\':decimal(decode_device_bins(dx)),\'cotangent_term\':decimal(decode_device_bins(dd)),\n    \'total\':decimal(decode_device_bins(total)),\'warning\':\'Algebraic split depends on chosen ordering; not unique causal percentages.\'})\n # Confirm the previously hand-calculated retained-difference percentage on TPU FP32.\n ga=rows[\'OLD\',0][\'raw_coordinate_gradient\'];gb=rows[\'NEW\',0][\'raw_coordinate_gradient\']\n va=exact[0,\'OLD\',\'OLD\'];vb=exact[0,\'NEW\',\'NEW\']\n retained=float(jax.jit(lambda a,b,c,d:100*(a-b)/(c-d))(*[jnp.asarray(v,jnp.float32) for v in (float(va),float(vb),ga,gb)]))\n host=100*float(va-vb)/(ga-gb)\n assert abs(retained-host)<1e-3\n r[\'retained_difference_percent\']={\'device_FP32\':retained,\'host_high_precision_reference\':host,\n  \'absolute_tolerance_percentage_points\':.001,\'passed\':True,\'scope\':\'This archived seed-0 OLD/NEW probe pair only.\'}\n r[\'status\']=\'passed\';r[\'all_exact_checks_passed\']=True\n r[\'new_full_model_execution\']=False;r[\'kernel_universal_correctness_proven\']=False\n r[\'interpretation\']=\'Confirms the exact local calculations for the archived tensors using fresh device integer operations. Does not establish the exact unrounded full-LM gradient or the exclusive upstream cause.\'\n for name,items in [(\'exact-cross-contractions.csv\',csvrows),(\'local-float-replays.csv\',r[\'local_float_replays\']),(\'decomposition.csv\',r[\'decomposition\'])]:\n  with (a.out/name).open(\'w\',newline=\'\') as f:w=csv.DictWriter(f,fieldnames=list(items[0]));w.writeheader();w.writerows(items)\n np.savez_compressed(a.out/\'exact-device-bins.npz\',**{f\'s{s}_{a}_{b}\':np.asarray(v) for (s,a,b),v in saved_bins.items()})\n dump(a.out,r);print(\'ALL_EXACT_DEVICE_CHECKS_PASSED\',backend,\'retained_percent\',retained,flush=True)\n\nif __name__==\'__main__\':\n p=argparse.ArgumentParser();p.add_argument(\'--capture\',type=Path,required=True);p.add_argument(\'--out\',type=Path,required=True);p.add_argument(\'--allow-cpu\',action=\'store_true\');a=p.parse_args();a.out.mkdir(exist_ok=True,parents=True)\n r={\'status\':\'running\',\'controls\':[],\'cross_contractions\':[],\'decomposition\':[],\'local_float_replays\':[]}\n try:run(a,r)\n except BaseException as e:r.update(status=\'failed\',error=repr(e),traceback=traceback.format_exc());dump(a.out,r);print(r[\'traceback\'],flush=True);raise\n'
SOURCE_SHA='92d35b67fa2ed93b1d3169390d387e44fc1cc89695ba162df92ed575851b4f76'
capture=OUT/'original-full-LM-capture.zip';capture.write_bytes(base64.b64decode(CAPTURE_B64))
assert hashlib.sha256(capture.read_bytes()).hexdigest()=='dd6854993c17d8f8a7e3fffdd3804d4c1481636c84768bb62ec2641090b615b6'
worker=OUT/'tpu_exact_confirmation.py';worker.write_text(SOURCE,encoding='utf-8')
assert hashlib.sha256(worker.read_bytes()).hexdigest()==SOURCE_SHA
print('CAPTURE_AND_SOURCE_HASHES_VERIFIED')


In [ ]:
import threading,queue
status={'status':'running'}
command=[PY,'-u',str(worker),'--capture',str(capture),'--out',str(OUT)]
try:
 process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
 q=queue.Queue()
 def reader():
  try:
   for line in process.stdout:q.put(line)
  finally:q.put(None)
 threading.Thread(target=reader,daemon=True).start()
 with (OUT/'worker.log').open('w') as log:
  while True:
   try:line=q.get(timeout=25)
   except queue.Empty:print('WORKING: TPU initialization / compilation / exact audit',flush=True);continue
   if line is None:break
   print(line,end='',flush=True);log.write(line);log.flush()
 code=process.wait();status.update(status='completed' if code==0 else 'failed',exit_code=code)
except BaseException as e:
 status.update(status='interrupted_or_error',detail=repr(e))
 if 'process' in globals() and process.poll() is None:process.terminate();process.wait()
finally:(OUT/'execution-status.json').write_text(json.dumps(status,indent=2))
print(status)


In [ ]:
p=OUT/'confirmation.json'
if p.exists():
 r=json.loads(p.read_text());print('STATUS',r['status'],'BACKEND',r.get('backend'))
 print('EXACT CROSS CONTRACTIONS',len(r['cross_contractions']),'DECOMPOSITIONS',len(r['decomposition']))
 print('RETAINED DIFFERENCE',r.get('retained_difference_percent'))
 print(r.get('interpretation','Incomplete or failed; inspect worker.log'))
manifest={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.iterdir() if p.is_file()}
DEST=WORK.parent/('Atomic-gdn2-exact-TPU-confirmation-results-'+RUN_ID+'.zip')
with zipfile.ZipFile(DEST,'w',zipfile.ZIP_DEFLATED) as z:
 for n in manifest:z.write(OUT/n,n)
 z.writestr('manifest.json',json.dumps({'files':manifest},indent=2))
with zipfile.ZipFile(DEST) as z:assert all(hashlib.sha256(z.read(n)).hexdigest()==h for n,h in manifest.items())
print('ARCHIVE',DEST,'SHA256',hashlib.sha256(DEST.read_bytes()).hexdigest())
from google.colab import files
files.download(str(DEST))
